In [15]:
# 필요한 모듈과 라이브러리를 로딩합니다.
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import urllib.request, urllib
import time
import math
import numpy
import pandas as pd
import random
import os
import re
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage
import requests

print("=" * 80)
print(" ulta 미국쇼핑몰을 수집하는 웹크롤러입니다. ")
print("=" * 80)

# 파일명 지정
query_txt = 'ulta 미국쇼핑몰'

f_dir = "c:\\py_temp\\"


# 결과를 저장할 폴더명과 파일명을 설정하고 폴더를 생성하기
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

sec_name = '팀프로젝트'

save_dir = f_dir + s + '-' + query_txt + '-' + sec_name + '\\'
img_dir = save_dir + 'image\\'

os.makedirs(img_dir, exist_ok=True)
os.chdir(img_dir)

ff_name = save_dir + s + '-' + query_txt + '.txt'
fc_name = save_dir + s + '-' + query_txt + '-' + sec_name + '.csv'
fx_name = save_dir + s + '-' + query_txt + '-' + sec_name + '.xlsx'

 ulta 미국쇼핑몰을 수집하는 웹크롤러입니다. 


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

driver = webdriver.Chrome()  
driver.get("https://www.ulta.com/")
driver.maximize_window()
time.sleep(2)

# Shop 클릭
driver.find_element(By.XPATH, '//span[@class="pal-c-Button__text" and text()="Shop"]').click()
time.sleep(2)

# Skin Care 클릭
driver.find_element(By.XPATH, '//span[@class="pal-c-Button__text" and text()="Skin Care"]').click()
time.sleep(2)

# Face Moisturizer 클릭
driver.find_element(By.XPATH, '//span[@class="Text-ds Text-ds--body-3 Text-ds--left Text-ds--black" and text()="Face Moisturizer"]').click()
time.sleep(2)

In [18]:
products = driver.find_elements(By.XPATH, '//li[contains(@class, "ProductListingResults__productCard")]')

links = []
for product in products:
    if len(links) == 10:
        break
    
    # Sponsored 제외
    sponsored = product.find_elements(By.XPATH, './/span[contains(@class, "pal-c-Tag__messageText") and text()="Sponsored"]')
    if sponsored:
        continue
    
    # 제품명 Cream 필터
    try:
        product_name = product.find_element(By.XPATH, './/p[contains(@class, "pal-c-Text__style--label")]').text
    except:
        continue
    
    if "Cream" not in product_name:
        continue
    
    # 리뷰수 200개 이상 필터
    try:
        review_text = product.find_element(By.XPATH, './/span[@class="sr-only"]').text
        review_count = int(review_text.split(';')[1].strip().replace('reviews', '').strip())
        if review_count < 200:
            continue
    except:
        continue
    
    # 링크 저장
    try:
        link = product.find_element(By.XPATH, './/a[contains(@class, "pal-c-Link--absolute")]').get_attribute('href')
        links.append({'name': product_name, 'link': link})
    except:
        continue

print("=" * 50)
print("  Sponsored 제외하고 Cream 이 들어간 제품들")
print("=" * 50)
for i, item in enumerate(links, 1):
    print(f"\n[{i}]")
    print(f"  제품명 : {item['name']}")
    print(f"  링크   : {item['link']}")
print("\n" + "=" * 50)

  Sponsored 제외하고 Cream 이 들어간 제품들

[1]
  제품명 : The Dewy Skin Cream Line-Plumping Moisturizer
  링크   : https://www.ulta.com/p/dewy-skin-cream-line-plumping-moisturizer-pimprod2049087?sku=2634098

[2]
  제품명 : Revitalizing Supreme+ Youth Power Cream Moisturizer
  링크   : https://www.ulta.com/p/revitalizing-supreme-youth-power-cream-moisturizer-pimprod2030772?sku=2591237

[3]
  제품명 : Cicaplast Balm B5 Soothing Therapeutic Multi Purpose Cream
  링크   : https://www.ulta.com/p/cicaplast-balm-b5-soothing-therapeutic-multi-purpose-cream-pimprod2018263?sku=2570171

[4]
  제품명 : Ultra Facial Cream with Squalane
  링크   : https://www.ulta.com/p/ultra-facial-cream-with-squalane-pimprod2002804?sku=2540233

[5]
  제품명 : Lipikar AP+M Triple Repair Body Moisturizing Cream for Dry Skin
  링크   : https://www.ulta.com/p/lipikar-apm-triple-repair-body-moisturizing-cream-dry-skin-xlsImpprod17102349?sku=2521449

[6]
  제품명 : Moisturizing Gel Cream
  링크   : https://www.ulta.com/p/moisturizing-gel-cream-pimprod2034790

In [19]:
results = []

for i, item in enumerate(links, 1):
    driver.get(item['link'])
    time.sleep(3)

    # 제품명
    try:
        product_name = driver.find_element(By.XPATH, '//span[contains(@class, "Text-ds--title-5")]').text
    except:
        product_name = ''

    # 브랜드
    try:
        brand = driver.find_element(By.XPATH, '//span[contains(@class, "Text-ds--body-1")]//div[contains(@class, "pal-c-Link__label")]').text
    except:
        brand = ''

    # 가격
    try:
        price = driver.find_element(By.XPATH, '//span[contains(@class, "pal-c-Text__style--title-lg")]').text
    except:
        price = ''

    # 별점
    try:
        rating = driver.find_element(By.XPATH, '//div[contains(@class, "ReviewStars__Content")]//span[contains(@class, "Text-ds--body-3")]').text
    except:
        rating = ''

    # 리뷰수
    try:
        review_els = driver.find_elements(By.XPATH, '//span[contains(@class, "Text-ds--body-3")]')
        review_count = ''
        for el in review_els:
            if 'Reviews' in el.text:
                review_count = el.text.replace('Reviews', '').replace(',', '').strip()
                break
    except:
        review_count = ''

    # 성분정보
    try:
        ing_btn = driver.find_element(By.XPATH, '//button[@aria-label="Ingredients"]')
        driver.execute_script("arguments[0].scrollIntoView();", ing_btn)
        time.sleep(1)
        driver.execute_script("arguments[0].click();", ing_btn)
        time.sleep(2)
        accordions = driver.find_elements(By.XPATH, '//div[contains(@class, "pal-c-Accordion__body--inner")]')
        ingredients = accordions[2].text
    except:
        ingredients = ''

    # 이미지 저장
    try:
        img_url = driver.find_element(By.XPATH, '//div[contains(@class, "MediaWrapper__Image")]//img').get_attribute('src')
        img_name = f'{i}.jpg'
        img_data = requests.get(img_url).content
        with open(img_dir + img_name, 'wb') as f:
            f.write(img_data)
    except:
        img_url = ''
        img_name = ''

    results.append({
        '번호': i,
        '제품명': product_name,
        '브랜드': brand,
        '가격': price,
        '별점': rating,
        '리뷰수': review_count,
        '링크': item['link'],
        '성분정보': ingredients,
        '이미지파일': img_name
    })

    print(f"[{i}/{len(links)}] 완료 | {brand} - {product_name}")

print("\n=== 완료 ===")
for r in results:
    print("=" * 50)
    print(f"제품이름 : {r['제품명']}")
    print(f"제품브랜드: {r['브랜드']}")
    print(f"가격     : {r['가격']}")
    print(f"별점     : {r['별점']}")
    print(f"리뷰수   : {r['리뷰수']}")
    print(f"링크     : {r['링크']}")
    print(f"성분정보 : {r['성분정보']}")

[1/10] 완료 | TATCHA - The Dewy Skin Cream Line-Plumping Moisturizer
[2/10] 완료 | Estée Lauder - Revitalizing Supreme+ Youth Power Cream Moisturizer
[3/10] 완료 | La Roche-Posay - Cicaplast Balm B5 Soothing Therapeutic Multi Purpose Cream
[4/10] 완료 | Kiehl's Since 1851 - Ultra Facial Cream with Squalane
[5/10] 완료 | La Roche-Posay - Lipikar AP+M Triple Repair Body Moisturizing Cream for Dry Skin
[6/10] 완료 | BYOMA - Moisturizing Gel Cream
[7/10] 완료 | TATCHA - The Water Cream Oil-Free Pore Minimizing Moisturizer
[8/10] 완료 | IT Cosmetics - Confidence in a Cream Anti-Aging Hydrating Moisturizer
[9/10] 완료 | First Aid Beauty - Ultra Repair Cream Intense Hydration Moisturizer
[10/10] 완료 | Avène - Cicalfate+ Restorative Protective Cream

=== 완료 ===
제품이름 : The Dewy Skin Cream Line-Plumping Moisturizer
제품브랜드: TATCHA
가격     : $74.00
별점     : 4.6
리뷰수   : 1215
링크     : https://www.ulta.com/p/dewy-skin-cream-line-plumping-moisturizer-pimprod2049087?sku=2634098
성분정보 : Aqua/Water/Eau, Saccharomyces/Rice Fer

In [22]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36',
    'Referer': 'https://www.ulta.com/'
}

# 이미지 저장
for i, item in enumerate(links, 1):
    driver.get(item['link'])
    time.sleep(3)
    try:
        img_url = driver.find_element(By.XPATH, '//div[contains(@class, "MediaWrapper__Image")]//img').get_attribute('src')
        img_data = requests.get(img_url, headers=headers).content
        with open(os.path.join(img_dir, f'{i}.jpg'), 'wb') as f:
            f.write(img_data)
        print(f"[{i}] 완료: {len(img_data)} bytes")
    except Exception as e:
        print(f"[{i}] 실패: {e}")

# 엑셀 저장
import pandas as pd
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage

df = pd.DataFrame()
df['Ranking'] = [r['번호'] for r in results]
df['Image'] = ''
df['Product_Name'] = [r['제품명'] for r in results]
df['Brand'] = [r['브랜드'] for r in results]
df['Price($)'] = [r['가격'] for r in results]
df['Rating'] = [r['별점'] for r in results]
df['Review_Cnt'] = [r['리뷰수'] for r in results]
df['Ingredients'] = [r['성분정보'] for r in results]
df['URL'] = [r['링크'] for r in results]

df.to_csv(fc_name, encoding='utf-8-sig', index=False)
df.to_excel(fx_name, index=False, engine='openpyxl')

wb = load_workbook(fx_name)
ws = wb["Sheet1"]

ws.column_dimensions['B'].width = 25
for r in range(2, len(results) + 2):
    ws.row_dimensions[r].height = 100

for i in range(1, len(results) + 1):
    img_path = os.path.join(img_dir, f"{i}.jpg")
    if not os.path.exists(img_path):
        continue
    img = XLImage(img_path)
    img.width = 130
    img.height = 100
    ws.add_image(img, f"B{i+1}")

wb.save(fx_name)
print(f"완료!\n엑셀: {fx_name}\nCSV: {fc_name}")

[1] 완료: 1261527 bytes
[2] 완료: 1503049 bytes
[3] 완료: 365384 bytes
[4] 완료: 821086 bytes
[5] 완료: 268294 bytes
[6] 완료: 297834 bytes
[7] 완료: 971759 bytes
[8] 완료: 746993 bytes
[9] 완료: 1026371 bytes
[10] 완료: 261145 bytes
완료!
엑셀: c:\py_temp\2026-04-24-18-12-40-ulta 미국쇼핑몰-팀프로젝트\2026-04-24-18-12-40-ulta 미국쇼핑몰-팀프로젝트.xlsx
CSV: c:\py_temp\2026-04-24-18-12-40-ulta 미국쇼핑몰-팀프로젝트\2026-04-24-18-12-40-ulta 미국쇼핑몰-팀프로젝트.csv
